FBCNET
Idea: Pseudo Online Evaluation of Deep Learning Models for Motor Imagery Direction Decoding Task
•	On Subject-1 Calibration data, split the data into train and test and compute the performance on Sub-1 Online Session data. 
•	For Sub-N, (N is between 2 to 20) 
o	Train data: Append all Sub-(N-1) Calibration data and correctly classified MI trials of Online session data
o	Test data: Sub-N Online Session Data

In [1]:
import scipy.io
import sys
import numpy as np
import os
import glob
import torch
from scipy import signal 
from sklearn.model_selection import LeaveOneOut
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

current_module = sys.modules[__name__]


In [2]:
def load_mat_file(filepath):
    """ Load .mat file and return Xtr, Ytr always; Xte, Yte only if they are not NaN """
    mat_data = scipy.io.loadmat(filepath)
    
    Xtr = mat_data['Xtrain']
    Ytr = mat_data['Ytrain']
    Xte = mat_data['Xtest']
    Yte = mat_data['Ytest']
    Yte_fb = mat_data['Ytest_fb']
    
    # Check if Xte and Yte are NaN or entirely NaN arrays
    if np.isnan(Xte).all() or np.isnan(Yte).all() or np.isnan(Yte_fb).all():
        # If all values in Xte or Yte are NaN, return only Xtr and Ytr
        return Xtr, Ytr
    else:
        # Otherwise, return Xtr, Ytr, Xte, Yte
        return Xtr, Ytr, Xte, Yte, Yte_fb

def create_dataset(current_subject, base_path=None):
    """"
    Create training and test dataset for the current subjects.
    
    Parameters:
    current_subject (int): Subject number (1 to 20).
    base_path (str): The directory where .mat files are stored.

    Returns:
    X_train, Y_train, X_test, Y_test
    """
    Xtr_all = []
    Ytr_all = []
    for sub in range(1, current_subject+1):
        rel_path = f'data/S{sub:02d}_mitrials.mat'
        filepath = os.path.join(base_path, rel_path)

        mat_vars = load_mat_file(filepath)
        if len(mat_vars)==2:
            Xtr, Ytr = mat_vars
            Xtr_all.append(Xtr)
            Ytr_all.append(Ytr)

        else:
            Xtr, Ytr, Xte, Yte, Yte_fb = mat_vars
            Xtr_all.append(Xtr)
            Ytr_all.append(Ytr)
            if (sub == current_subject):
                continue
            else:
                Xtr_all.append(Xte)
                Ytr_all.append(Yte)
                # indx = np.where(Yte==Yte_fb)[0]
                # Xtr_all.append(Xte[indx, :, :])
                # Ytr_all.append(Yte[indx])
                
    X_train = np.concatenate(Xtr_all, axis=0)
    Y_train = np.concatenate(Ytr_all, axis=0)

    X_test = Xte
    Y_test = Yte
        
    return X_train, Y_train, X_test, Y_test

# Baseline Correction 
def baseline_correction(X, baseline_samples=500):
    n_trails, n_samples, n_channels = X.shape
    Xbc = np.zeros_like(X)
    for t in range(n_trails):
        Xbase = X[t, :baseline_samples-1,:]
        Xeeg = X[t, baseline_samples:, :]
        Xbc[t, baseline_samples:, :] = Xeeg - np.mean(Xbase, axis=0) #baseline correction
    
    Xnew = Xbc[:, baseline_samples:, :]
    return Xnew

# Preprocessing: Surface Laplacian, Bandpass Filter
def bandpass_filtering(X, fs=500, fcut=[0.5, 45], filt_order=5):
    n_trials, n_samples, n_channels = X.shape
    X1 = np.zeros_like(X)
    b,a = signal.butter(filt_order, fcut, fs=fs, btype = 'band', output='ba') 
    for t in range(n_trials):
        for c in range(n_channels):
            #Error here.
            raw_signal = X[t, :, c]
            filt_signal = signal.filtfilt(b, a, raw_signal)
            X1[t, :, c] = filt_signal
            # Xfilt[t, :, c] = signal.filtfilt(b, a, X[t, :, c])
    return X1

# Multi-band Filter Function without averaging over trials
def filter_eeg_multi_band(X, fs=500):
    # Define the frequency bands
    bands = [(4, 8), (8, 12), (12, 16), (16, 20), (20, 24), 
             (24, 28), (28, 32), (32, 36), (36, 40)]
    
    n_trials, n_samples, n_channels = X.shape
    n_bands = len(bands)
    
    # Initialize output array: n_trials x samples x channels x n_bands
    Xout = np.zeros((n_trials, n_samples, n_channels, n_bands))
    
    # Apply bandpass filtering for each band
    for i, (low_cut, high_cut) in enumerate(bands):
        # print(f"Filtering band {low_cut}-{high_cut} Hz")
        X_filt = bandpass_filtering(X, fs=fs, fcut=[low_cut, high_cut], filt_order=5)
        
        # Store the filtered data without averaging across trials
        Xout[:, :, :, i] = X_filt
    
    Xout = np.transpose(Xout, (0, 2, 1, 3))  # (trials, electrodes, samples, n_bands)
    return Xout

In [3]:
class Conv2dWithConstraint(nn.Conv2d):
    def __init__(self, *args, doWeightNorm = True, max_norm=1, **kwargs):
        self.max_norm = max_norm
        self.doWeightNorm = doWeightNorm
        super(Conv2dWithConstraint, self).__init__(*args, **kwargs)

    def forward(self, x):
        if self.doWeightNorm: 
            self.weight.data = torch.renorm(
                self.weight.data, p=2, dim=0, maxnorm=self.max_norm
            )
        return super(Conv2dWithConstraint, self).forward(x)
    
class LinearWithConstraint(nn.Linear):
    def __init__(self, *args, doWeightNorm = True, max_norm=1, **kwargs):
        self.max_norm = max_norm
        self.doWeightNorm = doWeightNorm
        super(LinearWithConstraint, self).__init__(*args, **kwargs)

    def forward(self, x):
        if self.doWeightNorm: 
            self.weight.data = torch.renorm(
                self.weight.data, p=2, dim=0, maxnorm=self.max_norm
            )
        return super(LinearWithConstraint, self).forward(x)

#%% Support classes for FBNet Implementation
class VarLayer(nn.Module):
    '''
    The variance layer: calculates the variance of the data along given 'dim'
    '''
    def __init__(self, dim):
        super(VarLayer, self).__init__()
        self.dim = dim

    def forward(self, x):
        return x.var(dim = self.dim, keepdim= True)

class StdLayer(nn.Module):
    '''
    The standard deviation layer: calculates the std of the data along given 'dim'
    '''
    def __init__(self, dim):
        super(StdLayer, self).__init__()
        self.dim = dim

    def forward(self, x):
        return x.std(dim = self.dim, keepdim=True)

class LogVarLayer(nn.Module):
    '''
    The log variance layer: calculates the log variance of the data along given 'dim'
    (natural logarithm)
    '''
    def __init__(self, dim):
        super(LogVarLayer, self).__init__()
        self.dim = dim

    def forward(self, x):
        return torch.log(torch.clamp(x.var(dim = self.dim, keepdim= True), 1e-6, 1e6))

class MeanLayer(nn.Module):
    '''
    The mean layer: calculates the mean of the data along given 'dim'
    '''
    def __init__(self, dim):
        super(MeanLayer, self).__init__()
        self.dim = dim

    def forward(self, x):
        return x.mean(dim = self.dim, keepdim=True)

class MaxLayer(nn.Module):
    '''
    The max layer: calculates the max of the data along given 'dim'
    '''
    def __init__(self, dim):
        super(MaxLayer, self).__init__()
        self.dim = dim

    def forward(self, x):
        ma ,ima = x.max(dim = self.dim, keepdim=True)
        return ma

class swish(nn.Module):
    '''
    The swish layer: implements the swish activation function
    '''
    def __init__(self):
        super(swish, self).__init__()

    def forward(self, x):
        return x * torch.sigmoid(x)

class FBCNet(nn.Module):
    # just a FBCSP like structure : chan conv and then variance along the time axis
    '''
        FBNet with seperate variance for every 1s. 
        The data input is in a form of batch x 1 x chan x time x filterBand
    '''
    def SCB(self, m, nChan, nBands, doWeightNorm=True, *args, **kwargs):
        '''
        The spatial convolution block
        m : number of sptatial filters.
        nBands: number of bands in the data
        '''
        return nn.Sequential(
                Conv2dWithConstraint(nBands, m*nBands, (nChan, 1), groups= nBands,
                                     max_norm = 2 , doWeightNorm = doWeightNorm,padding = 0),
                nn.BatchNorm2d(m*nBands),
                swish()
                )

    def LastBlock(self, inF, outF, doWeightNorm=True, *args, **kwargs):
        return nn.Sequential(
                LinearWithConstraint(inF, outF, max_norm = 0.5, doWeightNorm = doWeightNorm, *args, **kwargs),
                nn.LogSoftmax(dim = 1))

    def __init__(self, nChan, nTime, nClass = 2, nBands = 9, m = 32,
                 temporalLayer = 'LogVarLayer', strideFactor= 4, doWeightNorm = True, *args, **kwargs):
        super(FBCNet, self).__init__()

        self.nBands = nBands
        self.m = m
        self.strideFactor = strideFactor

        # create all the parrallel SCBc
        self.scb = self.SCB(m, nChan, self.nBands, doWeightNorm = doWeightNorm)
        
        # Formulate the temporal agreegator
        self.temporalLayer = current_module.__dict__[temporalLayer](dim = 3)

        # The final fully connected layer
        self.lastLayer = self.LastBlock(self.m*self.nBands*self.strideFactor, nClass, doWeightNorm = doWeightNorm)

    def forward(self, x):
        x = torch.squeeze(x.permute((0,4,2,3,1)), dim = 4)
        x = self.scb(x)
        x = x.reshape([*x.shape[0:2], self.strideFactor, int(x.shape[3]/self.strideFactor)])
        x = self.temporalLayer(x)
        x = torch.flatten(x, start_dim= 1)
        x = self.lastLayer(x)
        return x

In [4]:
# Function to compute accuracy
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == targets.squeeze()).sum().item()
            total += targets.size(0)
    
    accuracy = correct / total
    return accuracy


In [5]:
torch.manual_seed(0)
parent_dir = os.path.dirname(os.getcwd())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size = 32
num_epochs = 200
fs = 500

perf = dict()
for sub in range(20, 21):
    # print(f'Current Subject : S{sub:02d}')
    Xtr, Ytr, Xte, Yte = create_dataset(sub, base_path=parent_dir)
    
    X_train = baseline_correction(Xtr)
    X_train = filter_eeg_multi_band(X_train)

    X_test = baseline_correction(Xte)
    X_test = filter_eeg_multi_band(X_test)

    # Convert the data to PyTorch tensors
    # X_train shape: (n_trials, electrodes, samples, bands) -> (n_trials, 1, electrodes, samples, bands)
    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1).permute(0, 1, 2, 3, 4).to(device)
    Y_train_tensor = torch.tensor(Ytr, dtype=torch.long).to(device)  # Use long for classification

    # X_test shape: (n_trials, electrodes, samples, bands) -> (n_trials, 1, electrodes, samples, bands)
    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1).permute(0, 1, 2, 3, 4).to(device)
    Y_test_tensor = torch.tensor(Yte, dtype=torch.long).to(device)

    # Create TensorDataset and DataLoader for train and test datasets
    train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
    test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)


    n_channels = 27
    n_time = 2000
    n_bands = 9
    n_classes = 2
    model = FBCNet(nChan=n_channels, nTime=n_time, nClass=n_classes, nBands=n_bands).to(device)

    criterion = nn.CrossEntropyLoss().to(device)  # Move loss function to GPU if needed
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # Training loop
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            # outputs = model(inputs.permute(0, 2, 1))  # Permute to match Conv1D input shape
            loss = criterion(outputs, targets.squeeze())
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
    
    # Compute accuracy on the test dataset
    accuracy = compute_accuracy(model, test_loader, device)
    perf[f'Sub{sub}'] = accuracy
    
    print(f'Test Subject: S{sub}, Test Accuracy: {accuracy:.4f}')

print(perf)

Test Subject: S20, Test Accuracy: 0.5417
{'Sub20': 0.5416666666666666}


In [ ]:
Test Subject: S8, Test Accuracy: 0.3750
Test Subject: S9, Test Accuracy: 0.5208
Test Subject: S10, Test Accuracy: 0.4375
Test Subject: S11, Test Accuracy: 0.5000
Test Subject: S12, Test Accuracy: 0.5208
Test Subject: S12, Test Accuracy: 0.5833
Test Subject: S13, Test Accuracy: 0.6458
Test Subject: S14, Test Accuracy: 0.4167
Test Subject: S15, Test Accuracy: 0.5208
Test Subject: S16, Test Accuracy: 0.5000
Test Subject: S17, Test Accuracy: 0.5000
Test Subject: S18, Test Accuracy: 0.4583
Test Subject: S19, Test Accuracy: 0.5000
Test Subject: S20, Test Accuracy: 0.5417




In [ ]:
mean_acc = list(perf.values())
print(mean_acc)
print(f'Average Accuracy: {np.mean(mean_acc)}')

In [ ]:
rel_path = f'data/S{8:02d}_mitrials.mat'
parent_dir = os.path.dirname(os.getcwd())
filepath = os.path.join(parent_dir, rel_path)
mat_vars = load_mat_file(filepath)
Xtr, Ytr, Xte, Yte, Yte_fb = mat_vars

X_train = baseline_correction(Xtr)
X_train = filter_eeg_multi_band(X_train)
print(f'Shape of X: {X_train.shape}')

X_test = baseline_correction(Xte)
X_test = filter_eeg_multi_band(X_test)
print(f'Shape of X: {X_test.shape}')


In [9]:
torch.manual_seed(0)
parent_dir = os.path.dirname(os.getcwd())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size = 32
num_epochs = 10
fs = 500

n_channels = 27
n_time = 2000
n_bands = 9
n_classes = 2
model = FBCNet(nChan=n_channels, nTime=n_time, nClass=n_classes, nBands=n_bands).to(device)


In [ ]:
criterion = nn.CrossEntropyLoss().to(device)  # Move loss function to GPU if needed
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Training loop
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        # outputs = model(inputs.permute(0, 2, 1))  # Permute to match Conv1D input shape
        loss = criterion(outputs, targets.squeeze())
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

# Compute accuracy on the test dataset
accuracy = compute_accuracy(model, test_loader, device)
print(f' Accuracy: {accuracy}')